# IMPORTS

In [ ]:
%load_ext autoreload

import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import numpy as np
import jax.numpy as jnp

import tol_colors as tc
import matplotlib.pyplot as plt
import plotting_tools.basic_plotting_tools as pts
colors = pts.set_plot_style(default_cmap=tc.sunset)
%matplotlib widget

In [ ]:
import differential_geometry.manifolds as manifolds
import plotting_tools.plot_manifolds as plot_manifolds

import differential_geometry.charts as charts

import differential_geometry.fields as fields
import differential_geometry.covectors_new as covectors

import differential_geometry.bundles_new as bundles
import plotting_tools.plot_bundles_new as plot_bundles

# 1D-mfd (ring) embedded in 3D: tangent and cotangent bundles together

Same ring as in `2_6_tangent_bundle.ipynb`, now with **both** bundles over it:

$$\left( T\mathcal{M}, \mathcal{O}_{T\mathcal{M}}, \mathcal{A}_{T\mathcal{M}} \right)_{2d}
\xrightarrow{\;\pi\;} \left( \mathcal{M}, \mathcal{O}, \mathcal{A}_{C^\infty} \right)_{d}
\xleftarrow{\;\pi^*\;}
\left( T^*\mathcal{M}, \mathcal{O}_{T^*\mathcal{M}}, \mathcal{A}_{T^*\mathcal{M}} \right)_{2d}.$$

Each bundle is drawn with its own fibre direction: the tangent fibres vertically (the cylinder,
as before) and the cotangent fibres radially, orthogonal to both the ring and the vertical.
Because the two directions are transverse, a pair $(v, \mathsf{X}_p) \in T_p\mathcal{M} \times
T_p^*\mathcal{M}$ can be drawn as a single point: start at $p$, move vertically by the vector and
then radially by the covector. The orange sheet over $p$ is the set of all such pairs, i.e. the
plane $T_p\mathcal{M} \times T_p^*\mathcal{M}$, attached along the tangent fibre.

On top of that we place:

- a vector field $\mathcal{X} \in \Gamma T\mathcal{M}$ — the blue section on the cylinder, with
  its value $v_{\gamma, p}$ drawn at a few points;
- a covector field $\Theta \in \Gamma T^*\mathcal{M}$ — drawn at those points as the orange arrow
  $\mathsf{X}_p = \Theta(p)$ starting at the tip of $v_{\gamma, p}$, so that the pair sits in its
  sheet, and globally as the orange curve $p \mapsto (\mathcal{X}_p, \Theta(p))$.

Forgetting the radial part of the orange curve gives back the blue section of $T\mathcal{M}$;
forgetting its vertical part gives the section of $T^*\mathcal{M}$ alone, which lives in the
horizontal annulus drawn by `cotangent_bundle.total_space` (set `draw_cotangent_annulus = True`
below to see it).

In [ ]:
# === Base manifold: a ring of radius R embedded in R^3 ===
RADIUS = 1.0

def ring_embedding(params):
    """Phi: theta -> (R cos theta, R sin theta, 0)."""
    angle = params[0]
    return jnp.stack([RADIUS * jnp.cos(angle), RADIUS * jnp.sin(angle), 0.0 * angle])

manifold = manifolds.Manifold(
    name=r"$\mathcal{M}$", dim=1, ambient_dim=3, embedding_func=ring_embedding)

# === A vector field (changes sign: up on the right, down on the left) ===
def vector_components(params):
    """Components w.r.t. d/d(theta)."""
    return jnp.array([0.1 + 0.5 * jnp.cos(params[0]) + 0.22 * jnp.sin(3.0 * params[0])])

# === A covector field (positive: all arrows point outward, as in the sketch) ===
def covector_components(params):
    """Components w.r.t. d(theta)."""
    return jnp.array([0.62 + 0.25 * jnp.cos(2.0 * params[0] + 0.5)])

field_X = fields.VectorField(manifold=manifold, vector_function=vector_components)
field_Theta = covectors.CovectorField(manifold=manifold, covector_function=covector_components)

# === Both bundles: tangent fibres drawn vertically, cotangent fibres radially (defaults) ===
tangent_bundle = bundles.TangentBundle(manifold=manifold, fibre_scale=1.0)
cotangent_bundle = bundles.CotangentBundle(manifold=manifold, fibre_scale=1.0)

# === Charts, used only for the checks below ===
chart_U = charts.Chart(
    name="U", manifold=manifold, boundary=(-1.0, 1.2),
    chart_map=lambda params: jnp.tan(params / 2.0),
    inverse_chart_map=lambda chart_coords: 2.0 * jnp.arctan(chart_coords))
chart_V = charts.Chart(
    name="V", manifold=manifold, boundary=(-0.6, 1.5),
    chart_map=lambda params: params + 0.4 * jnp.sin(params))

# === Base point p, the fibres drawn, and the windows of fibre values ===
p_param = jnp.array([0.0])
sample_angles = [0.0, 0.8, 2.35, jnp.pi, -2.3, -0.8]
tangent_range = (-0.95, 0.95)
cotangent_range = (0.0, 1.15)

## The cotangent bundle chart and the pairing

$$\xi^*_{\mathscr{X}}(\mathsf{X}_p) = \big( \mathscr{X}^i(p) \,;\, \mathsf{X}_p\big( (\partial / \partial \mathscr{X}^j)_p \big) \big),
\qquad
\left( \xi^*_{\mathscr{Y}} \circ {\xi^*_{\mathscr{X}}}^{-1} \right)(\alpha ; \beta)
= \big( \mathscr{T}_{\mathcal{V}\mathcal{U}}(\alpha) \,;\, (J_{\mathcal{U}\mathcal{V}})^k_{\ j}(p)\, \beta_k \big),
\qquad
\Theta\lceil \mathcal{X} \rfloor(p) = X_i(p)\, X^i(p) \text{ in any chart}.$$

In [ ]:
q_param = jnp.array([0.45])      # a point inside both chart domains

covector_at_q = field_Theta.evaluate_in_param_space(q_param)
bundle_point = cotangent_bundle.fibre(q_param, covector_at_q)

print("=== a point of T*M ===")
print("(theta ; w)             =", bundle_point)
print("pi*(theta ; w)          =", cotangent_bundle.projection(bundle_point), "   vs  q =", q_param)

print("\n=== cotangent bundle chart (covariant fibre block) ===")
chart_point = cotangent_bundle.chart_map(chart_U, bundle_point)
print("xi*_X(X_q)              =", chart_point)
print("  fibre block via field =", field_Theta.components_in_chart(chart_U, q_param))
print("xi*_X^{-1}(xi*_X(X_q))  =", cotangent_bundle.inverse_chart_map(chart_U, chart_point), " vs ", bundle_point)

print("\n=== transition map between cotangent bundle charts ===")
J_UV = covectors.chart_transition_jacobian(chart_U, chart_V, q_param)
transition_point = cotangent_bundle.chart_transition(chart_V, chart_U, chart_point)
print("fibre block of xi*_Y o xi*_X^{-1} =", transition_point[0, 1:])
print("(J_UV)^k_j beta_k                 =", J_UV.T @ chart_point[0, 1:])

print("\n=== the pairing Theta|X| is chart independent ===")
pairing = field_Theta.action_on_vector_field(field_X)
print("chart-free   :", pairing.evaluate_in_param_space(q_param[None, :])[0])
for name, chart in [("chart U", chart_U), ("chart V", chart_V)]:
    print(f"{name}      :", float(jnp.dot(field_Theta.components_in_chart(chart, q_param),
                                         field_X.components_in_chart(chart, q_param))))

## plot both bundles

Colours follow the hand-drawn figure: cyan for $T\mathcal{M}$, purple for the base manifold,
green for the tangent fibres and the projection, blue for the vector field and its values, orange
for everything cotangent. The fibres in the lists above are all drawn identically; the base point
$p$ only receives the labels.

In [ ]:
color_tangent_bundle = "deepskyblue"
color_base = "rebeccapurple"
color_fibre = "green"
color_vector = "mediumblue"
color_covector = "darkorange"

draw_cotangent_annulus = False

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")

# === Sheets T_pM x T*_pM over the sample points ===
for angle in sample_angles:
    plot_bundles.plot_fibre_pair_sheet(
        ax=ax, tangent_bundle=tangent_bundle, cotangent_bundle=cotangent_bundle,
        param_point=jnp.array([angle]), tangent_range=tangent_range,
        cotangent_range=cotangent_range, color=color_covector, alpha=0.10,
        edgecolor=color_covector, edge_linewidth=0.9
    )

# === Total space of TM (cylinder) and, optionally, of T*M (horizontal annulus) ===
plot_manifolds.plot_manifold(
    ax=ax, manifold=tangent_bundle.total_space, resolution=140,
    xmin=-jnp.pi, xmax=jnp.pi, ymin=tangent_range[0], ymax=tangent_range[1],
    color=color_tangent_bundle, alpha=0.16, edgecolor="none", label=None
)
if draw_cotangent_annulus:
    plot_manifolds.plot_manifold(
        ax=ax, manifold=cotangent_bundle.total_space, resolution=140,
        xmin=-jnp.pi, xmax=jnp.pi, ymin=cotangent_range[0], ymax=cotangent_range[1],
        color=color_covector, alpha=0.15, edgecolor="none", label=None
    )

# === Base manifold (the zero section of both bundles) ===
plot_bundles.plot_zero_section(
    ax=ax, bundle=tangent_bundle, param_range=(-jnp.pi, jnp.pi),
    color=color_base, linewidth=3.0
)

# === Fibres, vectors and covectors at the sample points ===
for angle in sample_angles:
    point = jnp.array([angle])
    vector_value = float(field_X.evaluate_in_param_space(point)[0])
    covector_value = float(field_Theta.evaluate_in_param_space(point)[0])

    plot_bundles.plot_fibre(
        ax=ax, bundle=tangent_bundle, param_point=point, fibre_range=tangent_range,
        color=color_fibre, linewidth=1.2, mutation_scale=9
    )
    plot_bundles.plot_vector_in_fibre(
        ax=ax, bundle=tangent_bundle, param_point=point, fibre_component=vector_value,
        color=color_vector, linewidth=2.0, mutation_scale=15,
        draw_base_point=False, draw_tip_marker=False
    )
    plot_bundles.plot_covector_in_fibre(
        ax=ax, cotangent_bundle=cotangent_bundle, param_point=point,
        covector_component=covector_value, tangent_bundle=tangent_bundle,
        vector_component=vector_value, color=color_covector, linewidth=2.0
    )

# === The two fields as curves ===
plot_bundles.plot_section_on_total_space(
    ax=ax, bundle=tangent_bundle, field=field_X, param_range=(-jnp.pi, jnp.pi),
    color=color_vector, linewidth=2.0
)
plot_bundles.plot_paired_section(
    ax=ax, tangent_bundle=tangent_bundle, cotangent_bundle=cotangent_bundle,
    vector_field=field_X, covector_field=field_Theta, param_range=(-jnp.pi, jnp.pi),
    color=color_covector, linewidth=2.0
)
if draw_cotangent_annulus:
    plot_bundles.plot_section_on_total_space(
        ax=ax, bundle=cotangent_bundle, field=field_Theta, param_range=(-jnp.pi, jnp.pi),
        color=color_covector, linewidth=1.5, alpha=0.6
    )

# === Labels at the base point p ===
vector_at_p = float(field_X.evaluate_in_param_space(p_param)[0])
covector_at_p = float(field_Theta.evaluate_in_param_space(p_param)[0])

plot_bundles.plot_vector_in_fibre(
    ax=ax, bundle=tangent_bundle, param_point=p_param, fibre_component=vector_at_p,
    color=color_vector, linewidth=2.0, mutation_scale=15,
    draw_base_point=False, draw_tip_marker=False,
    label=r"$v_{\gamma, p}$", label_offset=(0.08, -0.25, 0.0), label_fontsize=16
)
plot_bundles.plot_covector_in_fibre(
    ax=ax, cotangent_bundle=cotangent_bundle, param_point=p_param,
    covector_component=covector_at_p, tangent_bundle=tangent_bundle,
    vector_component=vector_at_p, color=color_covector, linewidth=2.0,
    label=r"$\mathsf{X}_p$", label_offset=(0.0, 0.0, 0.10), label_fontsize=16
)
plot_bundles.plot_projection_arrow(
    ax=ax, bundle=tangent_bundle, param_point=jnp.array([-2.3]), from_value=-0.9,
    to_value=-0.72, color=color_fibre, mutation_scale=10, label=r"$\pi$",
    label_offset=(0.1, 0.0, -0.05), label_fontsize=15
)

ax.text2D(0.47, 0.50, r"$\left( \mathcal{M}, \mathcal{O}, \mathcal{A}_{C^\infty} \right)_{d}$",
          transform=ax.transAxes, fontsize=13, color=color_base, ha="center")
ax.text2D(0.50, 0.17, r"$\left( T\mathcal{M}, \mathcal{O}_{T\mathcal{M}}, \mathcal{A}_{T\mathcal{M}} \right)_{2d}$",
          transform=ax.transAxes, fontsize=15, color=color_tangent_bundle, ha="center")
ax.text2D(0.97, 0.22, r"$\left( T^*\mathcal{M}, \mathcal{O}_{T^*\mathcal{M}}, \mathcal{A}_{T^*\mathcal{M}} \right)_{2d}$",
          transform=ax.transAxes, fontsize=15, color=color_covector, ha="right")
ax.text2D(0.40, 0.68, r"$\mathcal{X}$", transform=ax.transAxes, fontsize=18, color=color_vector)
ax.text2D(0.88, 0.47, r"$\Theta$", transform=ax.transAxes, fontsize=18, color=color_covector)

# === View ===
extent = RADIUS + cotangent_range[1] + 0.05
ax.set(xlim=(-extent, extent), ylim=(-extent, extent), zlim=tangent_range)
ax.set_box_aspect((2 * extent, 2 * extent, 1.35 * (tangent_range[1] - tangent_range[0])), zoom=1.25)
ax.view_init(elev=20.0, azim=-90.0)

plt.tight_layout()
plt.show()

fig.savefig("../saved_figures_and_videos/"+"cotangent_bundle.pdf", bbox_inches="tight", pad_inches=0.05, dpi=300)